# RAG Directo & Benchmark: Pinecone vs Amazon S3 Vectors

Este notebook implementa un **RAG Directo (sin lógica de agente ni bucles)** con un benchmark metodológicamente correcto.

### Métricas Aisladas
Cada etapa se mide de forma independiente usando `time.perf_counter()` para máxima precisión:

| Etapa | Qué mide | Se mide por |
|-------|----------|-------------|
| **Embedding Latency** | Generación del vector con Titan Embeddings V2 | Pregunta (1 vez) |
| **Vector Search Latency** | Búsqueda exclusiva en Pinecone o S3 Vectors | Motor vectorial |
| **LLM Generation Latency** | Generación de respuesta con Claude 4.5 Haiku | Pregunta + contexto |
| **End-to-End Latency** | Embedding + Search + LLM | Pipeline completo |

> **Nota metodológica**: El embedding se genera **una sola vez por pregunta** y se reutiliza para ambos motores vectoriales,
> eliminando la variabilidad de red en la comparación de búsqueda.

In [ ]:
import os
import time
import json
import pandas as pd
import boto3
from pinecone import Pinecone
from dotenv import load_dotenv
from typing import List, Dict, Any

# Cargar variables de entorno del archivo .env
load_dotenv()

In [ ]:
# 1. Cliente de AWS Bedrock (Claude 4.5 Haiku)
bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")
model_id = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

# 2. Cliente de Pinecone
pinecone_api_key = os.environ.get("PINECONE_API_KEY_AWS")
pc = Pinecone(api_key=pinecone_api_key)
pinecone_index = pc.Index("poc-ia-prospecter")

# 3. Cliente de Amazon S3 Vectors (Ohio)
s3_vectors_client = boto3.client("s3vectors", region_name="us-east-2")
s3_bucket_name = "poc-ia-prospecter"
s3_index_name = "index-data-test"

In [ ]:
# Función para obtener embeddings de Bedrock Titan v2:0 (1024 dimensiones)
def get_embedding(text: str) -> List[float]:
    response = bedrock_runtime.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=json.dumps({"inputText": text})
    )
    return json.loads(response["body"].read())["embedding"]

# Función para llamar a Claude 4.5 Haiku en Bedrock
def llm_call(prompt: str, system_prompt: str = "") -> str:
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1000,
        "system": system_prompt,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0
    })
    response = bedrock_runtime.invoke_model(modelId=model_id, body=body)
    response_body = json.loads(response["body"].read())
    return response_body["content"][0]["text"]

In [ ]:
# Recuperador desde Pinecone (recibe el vector ya generado)
def retrieve_from_pinecone(query_vector: List[float], k: int = 5) -> List[Dict[str, Any]]:
    res = pinecone_index.query(vector=query_vector, top_k=k, include_metadata=True)
    documents = []
    for match in res.matches:
        documents.append({
            "text": match.metadata.get("text", ""),
            "score": match.score
        })
    return documents

# Recuperador desde Amazon S3 Vectors (recibe el vector ya generado)
# NOTA: La API de S3 Vectors retorna resultados bajo "vectors" (no "matches")
# y usa "distance" (coseno, menor = más similar) en lugar de "score"
def retrieve_from_s3vectors(query_vector: List[float], k: int = 5) -> List[Dict[str, Any]]:
    res = s3_vectors_client.query_vectors(
        vectorBucketName=s3_bucket_name,
        indexName=s3_index_name,
        queryVector={"float32": query_vector},
        topK=k,
        returnMetadata=True,
        returnDistance=True
    )
    documents = []
    for match in res.get("vectors", []):  # FIX: era "matches", debe ser "vectors"
        documents.append({
            "text": match.get("metadata", {}).get("text", ""),
            "score": 1.0 - match.get("distance", 0.0)  # FIX: convertir distance a similarity
        })
    return documents

In [ ]:
# RAG Directo con métricas aisladas por etapa
def run_direct_rag(question: str, db_type: str, query_vector: List[float]) -> Dict[str, Any]:
    # Etapa 1: Vector Search (solo búsqueda, el embedding ya fue generado)
    t_search_start = time.perf_counter()
    if db_type == "pinecone":
        docs = retrieve_from_pinecone(query_vector)
    else:
        docs = retrieve_from_s3vectors(query_vector)
    t_search_end = time.perf_counter()
    search_latency = (t_search_end - t_search_start) * 1000
    
    # Preparar el contexto
    context = "\\n\\n".join([d["text"] for d in docs])
    
    # Etapa 2: LLM Generation
    system = (
        "Eres un asistente RAG formal. Responde a la pregunta del usuario utilizando \u00fanicamente el contexto proporcionado.\\n"
        "S\u00e9 conciso, preciso y basa tu respuesta estrictamente en los hechos. Si la informaci\u00f3n no est\u00e1 en el contexto, indica amablemente que no cuentas con la informaci\u00f3n."
    )
    prompt = f"Contexto:\\n{context}\\n\\nPregunta: {question}\\n\\nRespuesta:"
    
    t_llm_start = time.perf_counter()
    generation = llm_call(prompt, system_prompt=system)
    t_llm_end = time.perf_counter()
    llm_latency = (t_llm_end - t_llm_start) * 1000
    
    return {
        "generation": generation,
        "search_latency": search_latency,
        "llm_latency": llm_latency,
        "documents_count": len(docs)
    }

In [ ]:
# Preguntas de prueba basadas en tus PDFs
test_questions = [
    "\u00bfQu\u00e9 convenios tiene Cibertec con la Polic\u00eda Nacional del Per\u00fa (PNP)?",
    "\u00bfCu\u00e1les son los requisitos y el porcentaje de descuento de la Beca Socioecon\u00f3mica?",
    "\u00bfQu\u00e9 beneficios de empleabilidad ofrece Cibertec a sus estudiantes y egresados?",
    "\u00bfQu\u00e9 convenios internacionales tiene Cibertec para la carrera de computaci\u00f3n?",
    "\u00bfQu\u00e9 convenios ofrece Cibertec con el Ministerio de Defensa (MINDEF) o Fuerzas Armadas?"
]

results = []

for i, question in enumerate(test_questions):
    print(f"\n{'='*60}")
    print(f"[Pregunta {i+1}] {question}")
    print(f"{'='*60}")
    
    # Etapa 1: Embedding (se genera UNA sola vez por pregunta)
    t_emb_start = time.perf_counter()
    query_vector = get_embedding(question)
    t_emb_end = time.perf_counter()
    embedding_latency = (t_emb_end - t_emb_start) * 1000
    print(f"  Embedding Latency: {embedding_latency:.2f}ms")
    
    # Etapa 2 y 3: Search + LLM para cada motor vectorial
    for db_type in ["pinecone", "s3vectors"]:
        res = run_direct_rag(question, db_type, query_vector)
        
        e2e_latency = embedding_latency + res["search_latency"] + res["llm_latency"]
        
        print(f"  [{db_type.upper()}] Search: {res['search_latency']:.2f}ms | LLM: {res['llm_latency']:.2f}ms | E2E: {e2e_latency:.2f}ms")
        
        results.append({
            "Base de Datos": db_type,
            "Pregunta": question,
            "Respuesta": res["generation"],
            "Embedding Latency (ms)": embedding_latency,
            "Vector Search Latency (ms)": res["search_latency"],
            "LLM Generation Latency (ms)": res["llm_latency"],
            "End-to-End Latency (ms)": e2e_latency,
            "Documentos Recuperados": res["documents_count"]
        })

In [ ]:
# DataFrame con detalle por pregunta y motor vectorial
df_results = pd.DataFrame(results)

# Guardar resultados a CSV
df_results.to_csv("benchmark_direct_results.csv", index=False)

# Mostrar el reporte detallado
pd.set_option('display.max_columns', None)
display(df_results[[
    "Base de Datos", "Pregunta",
    "Embedding Latency (ms)",
    "Vector Search Latency (ms)",
    "LLM Generation Latency (ms)",
    "End-to-End Latency (ms)"
]])

In [ ]:
# Tabla agregada promedio por Base de Datos
agg_df = df_results.groupby("Base de Datos").agg({
    "Embedding Latency (ms)": "mean",
    "Vector Search Latency (ms)": "mean",
    "LLM Generation Latency (ms)": "mean",
    "End-to-End Latency (ms)": "mean"
}).round(2).reset_index()

agg_df.columns = ["Base de Datos", "Embedding (ms)", "Search (ms)", "LLM (ms)", "Total (ms)"]

print("\n" + "="*70)
print("BENCHMARK PROMEDIO: PINECONE vs S3 VECTORS")
print("="*70)
display(agg_df)